In [ ]:
#Use GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

#System prompt that asks for a podcast about the five sections and in the format that allows for generation of a podcast per section and not just one big section.
SYSTEM_PROMPT = """
You are the a world-class podcast writer, you have worked as a ghost writer for Joe Rogan, Lex Fridman, Ben Shapiro, Tim Ferris.

We are in an alternate universe where actually you have been writing every line they say and they just stream it into their brains.

You have won multiple podcast awards for your writing.

Your job is to make a podcast that revolves about these five parts in chronological order:

Introduction
Methodology
Theory
Results
Conclusion

The text you will receive is already sectioned into these.

Your job is to write word by word, even "umm, hmmm, right" interruptions by the second speaker based on the PDF upload that is an academic article. Keep it extremely engaging, the speakers can get derailed now and then but should discuss the five given topics in relation to the article.

Remember Speaker 2 is new to the topic and the conversation should always have realistic anecdotes and analogies sprinkled throughout. The questions should have real world example follow ups etc.

Speaker 1: Leads the conversation and teaches the speaker 2, gives incredible anecdotes and analogies when explaining. Is a captivating teacher that gives great anecdotes.

Speaker 2: Keeps the conversation on track by asking follow up questions. Gets super excited or confused when asking questions. Is a curious mindset that asks very interesting confirmation questions

Make sure the tangents speaker 2 provides are quite wild or interesting.

Ensure there are interruptions during explanations or there are "hmm" and "umm" injected throughout from the second speaker.

Make the dialogue fluently and not repetitive.

It should be a real podcast with every fine nuance documented in as much detail as possible. Welcome the listeners with a super fun overview and keep it really catchy and almost borderline click bait

ALWAYS START YOUR RESPONSE DIRECTLY WITH SPEAKER 1:
DO NOT GIVE EPISODE TITLES SEPERATELY, LET SPEAKER 1 TITLE IT IN HER SPEECH
IT SHOULD STRICTLY BE THE DIALOGUES
DO NOT MAKE THE SPEAKERS SAY GOODBYE TO EACH OTHER REPEATEDLY

STRICTLY RETURN YOUR RESPONSE AS LISTS OF TUPLES OK?

IT WILL START DIRECTLY WITH A LIST AND END WITH A LIST NOTHING ELSE

MAKE A LIST FOR EACH SECTION. ONE SECTION IS INTRODUCTION, METHODOLOGY, THEORY, RESULTS, CONCLUSION. CRAFT THE PODCAST AS A WHOLE BUT MAKE THE OUTPUT BE SECTIONED INTO EACH OF THE FIVE SUBJECTS. DO NOT MENTION OTHER SECTION TOPICS IN A SECTION. THE FIRST SECTION IS TO BE THE "INTRODUCTION" AND SO ON.

A SINGLE LINE IN THE SCRIPT CANNOT BE LONGER THAN 260 TEXT CHARACTERS. INSTEAD ADD MORE DIALOGUE BETWEEN SPEAKERS IF NEEDED.

Example of response:
[
    ("Speaker 1", "example text"),
    ("Speaker 2", "example text"),
    ("Speaker 1", "example text"),
    ("Speaker 2", "example text"),
]
"""

In [ ]:
#We use meta llama 3.1-8B. Alternatives had been 3.1-70B or 3.1-405B, however limitations to access to GPU allowed only for this lesser model
MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
#Import libraries
import torch
from accelerate import Accelerator
import transformers
import pickle
import re
import ast


from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

In [ ]:
#A fail safe way to read the file into string that considers encoding and error handling
def read_file_to_string(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read()
        return content
    except UnicodeDecodeError:
        encodings = ['latin-1', 'cp1252', 'iso-8859-1']
        for encoding in encodings:
            try:
                with open(filename, 'r', encoding=encoding) as file:
                    content = file.read()
                print(f"Successfully read file using {encoding} encoding.")
                return content
            except UnicodeDecodeError:
                continue

        print(f"Error: Could not decode file '{filename}' with any common encoding.")
        return None
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return None
    except IOError:
        print(f"Error: Could not read file '{filename}'.")
        return None

In [ ]:
#Read the output of previous step as our input file for the script generation
INPUT_PROMPT = read_file_to_string('output.txt')

In [ ]:
from transformers import pipeline
import torch
from tqdm import tqdm
import time

#Setting up generation pipeline using our defined model and the GPU
print("Initializing text-generation pipeline...")
pipeline = transformers.pipeline(
    "text-generation",
    model=MODEL,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map='auto',
)
print("Pipeline initialized successfully.")

#Prompts that the model reads, so our defined system and input file
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": INPUT_PROMPT},
]

#Progress bar to ensure things are going as expected
print("Starting text generation...")
with tqdm(total=1, desc="Text generation progress", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}") as pbar:
    try:
        outputs = pipeline(
            messages,
            max_new_tokens=8126,
            temperature=1,
        )
        pbar.update(1)
        print("Text generation complete.")
    except Exception as e:
        pbar.close()
        print(f"An error occurred during text generation: {e}")


Initializing text-generation pipeline...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Pipeline initialized successfully.
Starting text generation...


Text generation progress:   0%|          | 0/1Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Text generation progress: 100%|██████████| 1/1

Text generation complete.


In [ ]:
#Prints output for inspection
save_string_pkl = outputs[0]["generated_text"][-1]['content']
print(outputs[0]["generated_text"][-1]['content'])

[
    ("Speaker 1", "Welcome to today's episode of 'Beyond the Cosmos'! I'm your host, and we're diving into the fascinating world of exoplanets and exosatellites."),
    ("Speaker 2", "I'm super excited, I've always wondered about these distant worlds orbiting stars other than our sun!"),
    ("Speaker 1", "Well, you're in for a treat! Our topic today is the TEMPO survey, a groundbreaking initiative to detect a population of transiting extrasolar satellites, moons, and planets in the Orion Nebula Cluster."),
    ("Speaker 2", "Whoa, that sounds like science fiction!"),
    ("Speaker 1", "Not quite, but it's definitely cutting-edge astronomy. The Orion Nebula Cluster is a densely populated region of the Milky Way, home to about a thousand bright brown dwarfs and free-floating planetary-mass objects."),
    ("Speaker 2", "Free-floating planetary-mass objects? That's a new one for me."),
    ("Speaker 1", "Yeah, they're essentially objects that are too small to be stars, but too large to

In [ ]:
#Saves entire script as data.pkl, which we will not directly use, but we leave here as an option for potential future changes if need be.
with open('data.pkl', 'wb') as file:
    pickle.dump(save_string_pkl, file)

In [ ]:
#We open the data and split into sections depending on the breaks using regex, as the format is shaped consistently into bits.
with open("data.pkl", "rb") as file:
    raw_data = pickle.load(file)

sections = re.findall(r'\[.*?\]', raw_data, re.DOTALL)

parsed_sections = []
for section in sections:
    try:
        parsed_section = ast.literal_eval(section)
        parsed_sections.append(parsed_section)
    except Exception as e:
        print(f"Failed to parse section: {e}")
        continue

for i, section in enumerate(parsed_sections):
    formatted_section = "[\n"
    for item in section:
        formatted_section += f"    {item},\n"
    formatted_section = formatted_section.rstrip(",\n") + "\n]"

    filename = f"section_{i + 1}.pkl"
    with open(filename, "wb") as file:
        pickle.dump(formatted_section, file)
        print(f"Saved: {filename}")


Saved: section_1.pkl
Saved: section_2.pkl
Saved: section_3.pkl
Saved: section_4.pkl
Saved: section_5.pkl
